In [1]:
!pip install transformers torch torchaudio librosa soundfile huggingface_hub
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, classification_report
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoModel
import torch.nn as nn
import soundfile as sf
import librosa
from tqdm import tqdm
import matplotlib.pyplot as plt
import zipfile

In [2]:
GOOD_ZIP = "/content/хорошо.zip"
BAD_ZIP  = "/content/плохо.zip"

GOOD_DIR = Path("/content/хорошо")
BAD_DIR  = Path("/content/плохо")

GOOD_DIR.mkdir(parents=True, exist_ok=True)
BAD_DIR.mkdir(parents=True, exist_ok=True)

!ls -lh "/content"
!file "{GOOD_ZIP}"
!file "{BAD_ZIP}"

!apt-get -qq update
!apt-get -qq install -y p7zip-full

!7z x "{GOOD_ZIP}" -o"{GOOD_DIR}" -y
!7z x "{BAD_ZIP}"  -o"{BAD_DIR}"  -y

good_wavs = sorted(GOOD_DIR.rglob("*.wav"))
bad_wavs  = sorted(BAD_DIR.rglob("*.wav"))

total 662M
drwxr-xr-x 1 root root 4.0K May  6 13:32 sample_data
drwxr-xr-x 2 root root 4.0K May 10 10:11 плохо
-rw-r--r-- 1 root root 274M May 10 10:07 плохо.zip
drwxr-xr-x 2 root root 4.0K May 10 10:11 хорошо
-rw-r--r-- 1 root root 389M May 10 10:10 хорошо.zip
/content/хорошо.zip: Zip archive data, at least v2.0 to extract, compression method=deflate
/content/плохо.zip: Zip archive data, at least v2.0 to extract, compression method=deflate
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/                   1 file, 406858082 bytes (389 MiB)

Extracting archive: /con

In [3]:
df = pd.DataFrame({
    "path": [str(p) for p in good_wavs] + [str(p) for p in bad_wavs],
    "label": [0]*len(good_wavs) + [1]*len(bad_wavs)})
print("\ndf shape:", df.shape)
df.sample(5, random_state=42)


df shape: (2775, 2)


,path,label
879,/content/хорошо/eu.80be7cf3-5846-4608-981e-c04...,0
1989,/content/плохо/eu.1f5bfae8-63cf-48ec-bfd3-cdf1...,1
889,/content/хорошо/eu.81906517-c46d-4a8e-8827-d68...,0
912,/content/хорошо/eu.84aff07d-e9b6-45df-a229-3e3...,0
1596,/content/хорошо/eu.dac204d4-b6fe-4c79-b60a-868...,0


In [4]:
df.label.value_counts()

,count
label,
0,1870
1,905


In [5]:
df_train_tab, df_test_tab = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["label"].values
)

df_train_base, df_val_base = train_test_split(
    df_train_tab,
    test_size=0.2,
    random_state=42,
    stratify=df_train_tab["label"].values
)

df_train_base = df_train_base.reset_index(drop=True)
df_val_base   = df_val_base.reset_index(drop=True)
df_test_base  = df_test_tab.reset_index(drop=True)

print("Train:", len(df_train_base), "Val:", len(df_val_base), "Test:", len(df_test_base))
print("Train counts:\n", df_train_base["label"].value_counts())
print("Val counts:\n", df_val_base["label"].value_counts())
print("Test counts:\n", df_test_base["label"].value_counts())

Train: 1776 Val: 444 Test: 555
Train counts:
 label
0    1197
1     579
Name: count, dtype: int64
Val counts:
 label
0    299
1    145
Name: count, dtype: int64
Test counts:
 label
0    374
1    181
Name: count, dtype: int64


In [6]:
df_test_base

,path,label
0,/content/хорошо/eu.9b808a69-f082-425d-952c-017...,0
1,/content/хорошо/eu.ceb83047-6928-4a8a-b19f-080...,0
2,/content/хорошо/eu.7ecccdb0-abb6-4966-85f3-0f1...,0
3,/content/плохо/eu.ad64d2c8-0510-42c6-9440-9369...,1
4,/content/хорошо/eu.21c5bcbe-1f5b-401f-9421-f65...,0
...,...,...
550,/content/хорошо/eu.0932d188-e28c-44c4-901f-de5...,0
551,/content/хорошо/eu.6693d982-e7a8-4d56-b9bc-70e...,0
552,/content/хорошо/eu.f01dc51f-f235-46bb-99e4-719...,0
553,/content/плохо/eu.fd758618-c194-47cf-82dd-d592...,1


In [7]:
HF_REPO = "aksenovmr/wavlm_base_unfreeze4_speech_defects"
HF_FILENAME = "wavlm_base_unfreeze4_seed42.pt"

ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILENAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
sound_to_idx = ckpt["sound2id"]
idx_to_sound = {v: k for k, v in sound_to_idx.items()}
THR_BAD = float(ckpt["thr_bad"])

print(f"Звуки модели: {sorted(sound_to_idx.keys())}")
print(f"Порог thr_bad: {THR_BAD}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


wavlm_base_unfreeze4_seed42.pt:   0%|          | 0.00/378M [00:00<?, ?B/s]

cuda
Звуки модели: ['л', 'р', 'с', 'т', 'ц', 'ч', 'ш', 'щ']
Порог thr_bad: 0.2


In [8]:
class SoundDefectModel(nn.Module):
    def __init__(self, encoder_name: str, num_sounds: int, dropout: float = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden_size = self.encoder.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_sounds),
        )

    def forward(self, audio, attention_mask=None):
        out = self.encoder(input_values=audio, attention_mask=attention_mask)
        h = out.last_hidden_state

        if attention_mask is not None and hasattr(self.encoder, "_get_feature_vector_attention_mask"):
            feat_mask = self.encoder._get_feature_vector_attention_mask(
                h.shape[1], attention_mask
            )
            feat_mask = feat_mask.unsqueeze(-1).float()
            h_masked = h * feat_mask
            emb = h_masked.sum(dim=1) / feat_mask.sum(dim=1).clamp(min=1e-6)
        else:
            emb = h.mean(dim=1)

        logits = self.head(emb)
        p_defect = torch.sigmoid(logits)

        return p_defect, logits


model = SoundDefectModel(
    encoder_name=ckpt["encoder_name"],
    num_sounds=len(sound_to_idx),
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()
print("Модель загружена")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Модель загружена


In [9]:
SR = int(ckpt["sr"])
MAX_SECONDS = float(ckpt["max_seconds"])
MAX_LEN = int(SR * MAX_SECONDS)

def load_audio(path):
    try:
        x, file_sr = sf.read(path, always_2d=False)
        if x.ndim > 1:
            x = x.mean(axis=1)
        x = x.astype(np.float32)
        if file_sr != SR:
            x = librosa.resample(x, orig_sr=file_sr, target_sr=SR)
        return x
    except Exception:
        x, _ = librosa.load(path, sr=SR, mono=True)
        return x.astype(np.float32)


def pad_or_crop(x, max_len=MAX_LEN):
    if len(x) == 0:
        return np.zeros(max_len, dtype=np.float32)
    if len(x) > max_len:
        start = (len(x) - max_len) // 2
        return x[start:start + max_len]
    if len(x) < max_len:
        return np.pad(x, (0, max_len - len(x)))
    return x


def noisy_or_prob(probs):
    return 1.0 - np.prod(1.0 - np.clip(probs, 1e-7, 1 - 1e-7))


@torch.no_grad()
def predict_batch(paths, sounds_list):
    p_bads = []
    for path, sounds in zip(paths, sounds_list):
        x = pad_or_crop(load_audio(path), MAX_LEN)
        tensor = torch.tensor(x).unsqueeze(0).to(device)
        attn = (tensor != 0).long()
        p_defect, _ = model(tensor, attention_mask=attn)
        all_probs = p_defect.detach().cpu().numpy()[0]

        expected_indices = [sound_to_idx[s] for s in sounds if s in sound_to_idx]
        expected_probs = all_probs[expected_indices] if expected_indices else all_probs

        p_bad = noisy_or_prob(expected_probs)
        p_bads.append(float(p_bad))
    return np.array(p_bads)

In [10]:
def parse_sounds_from_filename(path: str) -> list:
    name = Path(path).stem
    if "__" not in name:
        return []
    labels_part = name.split("__")[-1]
    labels_part = labels_part.replace(" ", "").replace("-", "").replace("_", "")
    return list(labels_part)

In [11]:
p_test = []
BATCH = 16
paths = df_test_base["path"].tolist()
sounds_list = df_test_base["path"].apply(parse_sounds_from_filename).tolist()

for i in tqdm(range(0, len(paths), BATCH)):
    p_test.extend(predict_batch(paths[i:i+BATCH], sounds_list[i:i+BATCH]))

p_test = np.array(p_test)

test_df = df_test_base.copy()
test_df["p_bad"] = p_test
test_df["pred_label"] = (test_df["p_bad"] >= THR_BAD).astype(int)
test_df["true_label"] = test_df["label"].astype(int)
test_df["sounds"] = test_df["path"].apply(parse_sounds_from_filename)

test_df["error_type"] = "TN"
test_df.loc[(test_df.true_label == 1) & (test_df.pred_label == 1), "error_type"] = "TP"
test_df.loc[(test_df.true_label == 0) & (test_df.pred_label == 1), "error_type"] = "FP"
test_df.loc[(test_df.true_label == 1) & (test_df.pred_label == 0), "error_type"] = "FN"

print(test_df["error_type"].value_counts())

  0%|          | 0/35 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
100%|██████████| 35/35 [00:33<00:00,  1.05it/s]

error_type
TN    312
TP    154
FP     62
FN     27
Name: count, dtype: int64


In [12]:
test_df

,path,label,p_bad,pred_label,true_label,sounds,error_type
0,/content/хорошо/eu.9b808a69-f082-425d-952c-017...,0,0.024323,0,0,"[с, т, р, л, ц]",TN
1,/content/хорошо/eu.ceb83047-6928-4a8a-b19f-080...,0,0.272201,1,0,"[т, л, щ]",FP
2,/content/хорошо/eu.7ecccdb0-abb6-4966-85f3-0f1...,0,0.254088,1,0,"[с, т, р, л, ш]",FP
3,/content/плохо/eu.ad64d2c8-0510-42c6-9440-9369...,1,0.101049,0,1,"[с, т, р, л]",FN
4,/content/хорошо/eu.21c5bcbe-1f5b-401f-9421-f65...,0,0.031812,0,0,"[л, ш]",TN
...,...,...,...,...,...,...,...
550,/content/хорошо/eu.0932d188-e28c-44c4-901f-de5...,0,0.008797,0,0,"[т, р]",TN
551,/content/хорошо/eu.6693d982-e7a8-4d56-b9bc-70e...,0,0.002321,0,0,"[с, т, р]",TN
552,/content/хорошо/eu.f01dc51f-f235-46bb-99e4-719...,0,0.044042,0,0,"[ч, т, р]",TN
553,/content/плохо/eu.fd758618-c194-47cf-82dd-d592...,1,0.917993,1,1,"[с, л]",TP


In [13]:
all_sounds = sorted(sound_to_idx.keys())
results = []

for sound in all_sounds:
    mask = test_df["sounds"].apply(lambda s: sound in s)
    subset = test_df[mask]

    n_total = len(subset)
    n_fn = (subset["error_type"] == "FN").sum()
    n_fp = (subset["error_type"] == "FP").sum()
    n_bad = (subset["true_label"] == 1).sum()
    n_good = (subset["true_label"] == 0).sum()

    results.append({
        "sound":   sound,
        "total":   n_total,
        "n_bad":   n_bad,
        "n_good":  n_good,
        "FN":      n_fn,
        "FP":      n_fp,
        "FN_rate": round(n_fn / n_bad, 3) if n_bad > 0 else None,
        "FP_rate": round(n_fp / n_good, 3) if n_good > 0 else None,
    })

df_sounds = pd.DataFrame(results).set_index("sound")
df_sounds.sort_values("FN_rate", ascending=False)

,total,n_bad,n_good,FN,FP,FN_rate,FP_rate
sound,,,,,,,
щ,46,16,30,4,5,0.250,0.167
ч,129,42,87,8,12,0.190,0.138
ш,187,57,130,10,27,0.175,0.208
т,444,157,287,26,47,0.166,0.164
с,309,87,222,14,32,0.161,0.144
л,396,128,268,19,46,0.148,0.172
р,416,144,272,16,48,0.111,0.176
ц,77,15,62,1,8,0.067,0.129


In [14]:
def estimate_noise(path):
    try:
        y, _ = librosa.load(path, sr=SR)
        return float(np.percentile(np.abs(y), 10))
    except:
        return np.nan

def estimate_duration(path):
    try:
        y, _ = librosa.load(path, sr=SR)
        return len(y) / SR
    except:
        return np.nan

test_df["noise_level"] = test_df["path"].apply(estimate_noise)
test_df["duration_sec"] = test_df["path"].apply(estimate_duration)

summary = test_df.groupby("error_type")[["noise_level", "duration_sec"]].agg(
    ["mean", "median"]
)
summary

noise_level           duration_sec           
                  mean    median         mean     median
error_type                                              
FN            0.000616  0.000397     9.307146   8.600000
FP            0.000782  0.000381    11.308954   9.870000
TN            0.000665  0.000336     7.964006   7.059656
TP            0.000613  0.000305    13.951658  11.680000

In [15]:
test_df["n_sounds"] = test_df["sounds"].apply(len)

print(test_df.groupby("error_type")["n_sounds"].agg(["mean", "median", "count"]))
test_df.groupby(["n_sounds", "error_type"]).size().unstack(fill_value=0)

                mean  median  count
error_type                         
FN          3.629630     3.0     27
FP          3.629032     4.0     62
TN          3.631410     4.0    312
TP          3.558442     3.0    154


error_type,FN,FP,TN,TP
n_sounds,,,,
1,0,3,3,3
2,4,4,36,11
3,10,19,113,69
4,9,25,88,47
5,2,10,66,19
6,0,0,5,2
7,2,1,1,3


In [16]:
results = []
for sound in all_sounds:
    mask = test_df["sounds"].apply(lambda s: sound in s)
    subset = test_df[mask]

    n_bad  = (subset["true_label"] == 1).sum()
    n_good = (subset["true_label"] == 0).sum()

    auc = round(roc_auc_score(subset.true_label, subset.p_bad), 3) if n_bad > 0 and n_good > 0 else None

    results.append({
        "sound":   sound,
        "total":   len(subset),
        "n_bad":   int(n_bad),
        "n_good":  int(n_good),
        "F1":      round(f1_score(subset.true_label, subset.pred_label, zero_division=0), 3),
        "ROC-AUC": auc,
        "FN_rate": round((subset.error_type == "FN").sum() / n_bad, 3) if n_bad > 0 else None,
        "FP_rate": round((subset.error_type == "FP").sum() / n_good, 3) if n_good > 0 else None,
    })

df_by_sound = pd.DataFrame(results).set_index("sound")
df_by_sound.sort_values("F1", ascending=False)

,total,n_bad,n_good,F1,ROC-AUC,FN_rate,FP_rate
sound,,,,,,,
р,416,144,272,0.800,0.933,0.111,0.176
т,444,157,287,0.782,0.915,0.166,0.164
ч,129,42,87,0.773,0.904,0.190,0.138
л,396,128,268,0.770,0.913,0.148,0.172
с,309,87,222,0.760,0.924,0.161,0.144
ц,77,15,62,0.757,0.977,0.067,0.129
щ,46,16,30,0.727,0.883,0.250,0.167
ш,187,57,130,0.718,0.910,0.175,0.208
